In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score,auc,precision_score, recall_score, f1_score

In [2]:
def returnSolution(value,criterea):
    if value >= criterea:
        return 1
    else :
        return 0

In [3]:
X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_test = pd.read_parquet("../data/processed/y_test.parquet")

In [4]:
cols_woes = []
cols_norm = []
for col in X_test.columns:
    if "_woe" in col:
        cols_woes.append(col)
    else:
        cols_norm.append(col)

In [5]:
modelBaseline = joblib.load("../assets/models/baselineModelNorm.joblib")
modelBaselineWoes = joblib.load("../assets/models/baselineModelWoes.joblib")

In [6]:
modelLgb = joblib.load("../assets/models/lgb_norm.joblib")
modelLgbWoe = joblib.load("../assets/models/lgb_woe.joblib")

In [7]:
modelXgm = joblib.load("../assets/models/xgb_norm.joblib")
modelXgmWoe = joblib.load("../assets/models/xgb_woe.joblib")

In [8]:
probaBaseLine = modelBaseline.predict_proba(X_test[cols_norm])[:, 1]
probaBaseLineWoe = modelBaselineWoes.predict_proba(X_test[cols_woes])[:, 1]

In [9]:
probaLgb = modelLgb.predict_proba(X_test[cols_norm])[:, 1]
probaLgbWoe = modelLgbWoe.predict_proba(X_test[cols_woes])[:, 1]

In [10]:
probaXgm = modelXgm.predict_proba(X_test[cols_norm])[:, 1]
probaXgmWoe = modelXgmWoe.predict_proba(X_test[cols_woes])[:, 1]

In [11]:
y_test["BaseLine"] = probaBaseLine
y_test["BaseLineWoe"] = probaBaseLineWoe
y_test["Lgb"] = probaLgb
y_test["LgbWoe"] = probaLgbWoe
y_test["Xgm"] = probaXgm
y_test["XgmWoe"] = probaXgmWoe

In [12]:
arrayx = ["BaseLine", "BaseLineWoe", "Lgb", "LgbWoe", "Xgm", "XgmWoe"]

In [13]:
array_cut = []
float_range = [x * 0.05 for x in range(1, 20)]
for v in float_range:   
    array_cut.append(float(f"{v:.2f}"))

In [35]:
prec_dict = {}
recall_dict = {}
f1_dict = {}
for col in arrayx:
    prec_array = []
    recall_array = []
    f1_array = []
    for cut in array_cut:
        prev = y_test[col].apply(lambda x:returnSolution(x,cut))
        real = y_test["Target"]
        metric_prec = precision_score(real,prev)
        metric_recall = recall_score(real,prev)
        metric_f1 = f1_score(real,prev)
        prec_array.append(metric_prec)
        recall_array.append(metric_recall)
        f1_array.append(metric_f1)
    prec_dict[f"{col}"] = prec_array
    recall_dict[f"{col}"] = recall_array
    f1_dict[f"{col}"] = f1_array


d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

In [36]:
df_pred = pd.DataFrame(prec_dict)

In [37]:
df_pred

,BaseLine,BaseLineWoe,Lgb,LgbWoe,Xgm,XgmWoe
0,0.102251,0.185073,0.182854,0.179427,0.183321,0.178949
1,0.150092,0.301592,0.278415,0.274410,0.275547,0.279450
2,0.210652,0.361188,0.348369,0.343860,0.350327,0.343456
3,0.294146,0.398907,0.395044,0.387949,0.395295,0.386221
4,0.488372,0.433892,0.429935,0.426211,0.429859,0.426587
5,0.566787,0.460886,0.463542,0.454392,0.464894,0.454111
6,0.550336,0.480492,0.510628,0.481928,0.503675,0.485922
7,0.582418,0.505398,0.553154,0.531521,0.540211,0.525057
8,0.527778,0.548478,0.573661,0.558632,0.583819,0.554054
9,0.549020,0.575047,0.612179,0.592771,0.611594,0.592075
